# 02 - Data Preprocessing

**Project:** DetectIt – Intelligent Web Threat Detection Using Machine Learning

**Goal:** In the EDA notebook, we found a few problems in the dataset and made a plan to fix them. In this notebook, we follow that plan step by step. At the end, we save one training set and one test set, so every team member can train their models on the same data.

**Input:** `data/PhiUSIIL_Phishing_URL_Dataset.csv`

**Output:** `data/processed/train.csv`, `data/processed/test.csv`, `data/processed/feature_names.json`

## Setup

We load the libraries and keep all settings in one place. The columns we remove later were chosen in the EDA notebook. We also load our own feature functions from `src/features.py`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append(str(Path("..").resolve()))
from src.features import build_url_features, build_page_features

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/PhiUSIIL_Phishing_URL_Dataset.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = "label"
TEXT_COLS = ["FILENAME", "URL", "Domain", "TLD", "Title"]
REDUNDANT_COLS = ["URLTitleMatchScore", "URLLength"]

In [2]:
df = pd.read_csv(DATA_PATH)
initial_shape = df.shape
print(f"Loaded dataset: {initial_shape[0]:,} rows | {initial_shape[1]} columns")

Loaded dataset: 235,795 rows | 56 columns


## 1. Data Validation

Before we change anything, we check that the values make sense. A count, like the number of images on a page, can never be negative. A yes/no column should only have 0 or 1. The label should also only be 0 or 1.

In [3]:
numeric_cols = df.select_dtypes(include="number").columns.drop(TARGET)
binary_cols = [col for col in numeric_cols if df[col].nunique() == 2]

negative_cols = [col for col in numeric_cols if (df[col] < 0).any()]
invalid_binary_cols = [col for col in binary_cols if not set(df[col].unique()) <= {0, 1}]
invalid_labels = set(df[TARGET].unique()) - {0, 1}

print(f"Columns with negative values: {negative_cols}")
print(f"Binary columns with values other than 0/1: {invalid_binary_cols}")
print(f"Unexpected label values: {invalid_labels}")

Columns with negative values: []
Binary columns with values other than 0/1: []
Unexpected label values: set()


## 2. Remove Duplicate URLs

In the EDA, we found 425 URLs that appear twice. If we keep them, one copy could go into the training set and the other into the test set. The model would then be tested on a URL it has already seen, and the results would look better than they really are. So we keep the first copy of each URL and remove the second.

In [4]:
duplicate_count = df["URL"].duplicated().sum()
df = df.drop_duplicates(subset="URL", keep="first").reset_index(drop=True)

print(f"Removed {duplicate_count} duplicate URLs")
print(f"Rows remaining: {len(df):,}")

Removed 425 duplicate URLs
Rows remaining: 235,370


## 3. Feature Engineering

Our coordinator suggested creating new features from the existing data. We create two groups of features:

- **URL features (18):** made only from the URL text, such as the number of hyphens, suspicious words like "login" or "verify", and how random the characters look (entropy). Because they only need the URL, we can also create them for the second dataset and inside the DetectIt app.
- **Page features (8):** made by combining existing page columns, such as the share of links that go to other websites, and a simple score that counts how many signs of a real website a page has.

The code for these features is in `src/features.py`, so the same code is used everywhere.

In [5]:
url_features = build_url_features(df["URL"])
page_features = build_page_features(df)
df = pd.concat([df, url_features, page_features], axis=1)

ENGINEERED_COLS = url_features.columns.tolist() + page_features.columns.tolist()
print(f"Created {len(url_features.columns)} URL features and {len(page_features.columns)} page features")
print(f"Columns now: {df.shape[1]}")

Created 18 URL features and 8 page features
Columns now: 82


In [6]:
is_phishing = df[TARGET].eq(0).astype(int)
engineered_corr = df[ENGINEERED_COLS].corrwith(is_phishing).sort_values(key=abs, ascending=False)
engineered_corr.round(3).to_frame("Correlation with Phishing")

,Correlation with Phishing
PageQualityScore,-0.853
HasHTTPSScheme,-0.614
HasSensitiveForm,-0.506
NoOfSlashesInURL,0.482
URLEntropy,0.396
HasFinancialKeyword,-0.388
NoOfDigitsInHost,0.344
PathDepth,0.333
ExternalRefRatio,-0.328
HostHasHyphen,0.308


In [7]:
numeric_features = df.select_dtypes(include="number").columns.drop(TARGET)
corr_matrix = df[numeric_features].corr()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
pairs = upper.stack().dropna()

involves_new = [a in ENGINEERED_COLS or b in ENGINEERED_COLS for a, b in pairs.index]
repeated_pairs = pairs[involves_new][pairs[involves_new].abs() > 0.95].sort_values(key=abs, ascending=False)
repeated_pairs.index.names = ["Feature 1", "Feature 2"]

print(f"Pairs with a new feature and |correlation| > 0.95: {len(repeated_pairs)}")
repeated_pairs.rename("Correlation").round(3).reset_index()

Pairs with a new feature and |correlation| > 0.95: 6


,Feature 1,Feature 2,Correlation
0,URLLength,FullURLLength,1.000
1,DomainLength,HostLength,1.000
2,IsHTTPS,HasHTTPSScheme,0.994
3,NoOfSubDomain,HostSubdomainCount,0.986
4,IsDomainIP,HostIsIP,0.976
5,NoOfLettersInURL,FullURLLength,0.956


### Findings: Feature Engineering

- We created 26 new features: 18 from the URL text and 8 from the page columns.
- **`PageQualityScore`** is the strongest new feature (-0.853). It is almost as strong as `URLSimilarityIndex`, the best original feature. Phishing pages have far fewer of the signs that a real website usually has, such as a favicon, a description, copyright text and social media links.
- **`HasSensitiveForm`** (-0.506) and **`HasFinancialKeyword`** (-0.388) are negative, which means these are more common on legitimate pages. This makes sense, because real banking, shopping and login pages naturally contain forms and payment words.
- Among the URL features, phishing URLs have more slashes (0.482), more random characters (`URLEntropy`, 0.396), more digits in the domain (0.344), deeper paths (0.333) and more hyphens in the domain (0.308).
- Some features, such as `TitleWordCount` and `HostSubdomainCount`, have almost no linear link to the label. We still keep them, because tree-based models can find patterns that correlation cannot show.
- **5 of our URL features repeat an original feature** (correlation above 0.95): `FullURLLength`, `HostLength`, `HasHTTPSScheme`, `HostSubdomainCount` and `HostIsIP`. This is a good sign, because it shows our own URL code gives almost the same values as the dataset's own features. For this dataset, we remove these 5 copies and keep the original features. We still keep them in `src/features.py`, because the URL-only model for the second dataset and the app needs them.

## 4. Remove Text Columns

ML models can only work with numbers, so we now remove the five text columns: `FILENAME`, `URL`, `Domain`, `TLD` and `Title`. We do not lose useful information here. The dataset already turns these columns into numeric features (for example, `TLDLegitimateProb` comes from the TLD), and in the previous step we created our own features from the URL and the title.

Before removing the `URL` column, we keep a copy of it. We will use it later to check that no URL appears in both the training and test sets.

In [8]:
urls = df["URL"].copy()
df = df.drop(columns=TEXT_COLS)

print(f"Removed {len(TEXT_COLS)} text columns: {TEXT_COLS}")
print(f"Columns remaining: {df.shape[1]}")

Removed 5 text columns: ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
Columns remaining: 77


## 5. Remove Repeated Features

Some features carry almost the same information as another feature (correlation above 0.95). Keeping both does not add anything new, and it can confuse models like Logistic Regression and Naive Bayes. We remove:

- **2 original features** found in the EDA: `URLTitleMatchScore` and `URLLength`.
- **5 new features** that copy an original feature, found in Section 3: `FullURLLength`, `HostLength`, `HasHTTPSScheme`, `HostSubdomainCount` and `HostIsIP`.

In [9]:
repeated_new_features = sorted({b if b in ENGINEERED_COLS else a for a, b in repeated_pairs.index})
cols_to_drop = REDUNDANT_COLS + repeated_new_features
df = df.drop(columns=cols_to_drop)

print(f"Removed {len(cols_to_drop)} repeated features: {cols_to_drop}")
print(f"Columns remaining: {df.shape[1]}")

Removed 7 repeated features: ['URLTitleMatchScore', 'URLLength', 'FullURLLength', 'HasHTTPSScheme', 'HostIsIP', 'HostLength', 'HostSubdomainCount']
Columns remaining: 70


## 6. Change the Label

In the original dataset, 1 means legitimate and 0 means phishing. We swap this so that **1 = Phishing** and **0 = Legitimate**. Phishing is what we want to find, so it should be the positive class. This way, precision and recall will directly tell us how well the models catch phishing URLs.

In [10]:
df[TARGET] = (df[TARGET] == 0).astype(int)

LABEL_NAMES = {0: "Legitimate", 1: "Phishing"}
df[TARGET].map(LABEL_NAMES).value_counts()

label
Legitimate    134850
Phishing      100520
Name: count, dtype: int64

## 7. Split the Data into Training and Test Sets

We split the data into 80% for training and 20% for testing. The model learns from the training set, and we use the test set only at the end to see how well it works on URLs it has never seen.

We use a stratified split, so both sets keep the same mix of phishing and legitimate URLs. We also fix `random_state = 42`, so every team member gets exactly the same split.

In [11]:
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test, urls_train, urls_test = train_test_split(
    X, y, urls,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training set: {X_train.shape[0]:,} rows | {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]:,} rows | {X_test.shape[1]} features")

Training set: 188,296 rows | 69 features
Test set:     47,074 rows | 69 features


## 8. Check the Split

We check two things. First, the share of phishing URLs should be almost the same in the full data, the training set and the test set. Second, no URL should appear in both the training set and the test set. If both checks pass, there is no data leakage.

In [12]:
class_ratio = pd.DataFrame({
    "Full data": y.value_counts(normalize=True),
    "Training set": y_train.value_counts(normalize=True),
    "Test set": y_test.value_counts(normalize=True),
}).rename(index=LABEL_NAMES) * 100

class_ratio.round(2)

,Full data,Training set,Test set
label,,,
Legitimate,57.29,57.29,57.29
Phishing,42.71,42.71,42.71


In [13]:
shared_urls = set(urls_train) & set(urls_test)
print(f"URLs found in both training and test sets: {len(shared_urls)}")

URLs found in both training and test sets: 0


## 9. Save the Processed Data

We save the training and test sets as CSV files, so every team member can load the same data in the model notebooks. We also save:

- the URLs of each set, which we need for the URL-only model tested on the second dataset (Experiment 2), and
- the list of feature names, split into original and new features, so we can compare models with and without our new features.

In [14]:
train_df = X_train.assign(**{TARGET: y_train})
test_df = X_test.assign(**{TARGET: y_test})

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)
urls_train.to_csv(PROCESSED_DIR / "train_urls.csv", index=False)
urls_test.to_csv(PROCESSED_DIR / "test_urls.csv", index=False)

feature_groups = {
    "all": X_train.columns.tolist(),
    "original": [col for col in X_train.columns if col not in ENGINEERED_COLS],
    "engineered": [col for col in X_train.columns if col in ENGINEERED_COLS],
}
with open(PROCESSED_DIR / "feature_names.json", "w") as f:
    json.dump(feature_groups, f, indent=2)

print(f"Saved files to {PROCESSED_DIR.resolve()}")
print({name: len(cols) for name, cols in feature_groups.items()})

Saved files to D:\DetectIt\data\processed
{'all': 69, 'original': 48, 'engineered': 21}


In [15]:
print(f"Before: {initial_shape[0]:,} rows | {initial_shape[1]} columns")
print(f"After:  {len(df):,} rows | {df.shape[1]} columns ({X.shape[1]} features + 1 label)")

Before: 235,795 rows | 56 columns
After:  235,370 rows | 70 columns (69 features + 1 label)


## Preprocessing Summary

| Step | What We Did | Result |
|---|---|---|
| 1. Data validation | Checked for negative counts, wrong binary values and wrong labels | No problems found |
| 2. Duplicate URLs | Removed the second copy of each repeated URL | 425 rows removed (235,795 → 235,370) |
| 3. Feature engineering | Created 18 URL features and 8 page features | 26 new features |
| 4. Text columns | Removed `FILENAME`, `URL`, `Domain`, `TLD` and `Title` | 5 columns removed |
| 5. Repeated features | Removed 2 original and 5 new features that copy another feature | 7 columns removed |
| 6. Label | Changed the label to 1 = Phishing, 0 = Legitimate | 100,520 phishing and 134,850 legitimate URLs |
| 7. Split | Stratified 80/20 split with `random_state = 42` | 188,296 training rows and 47,074 test rows |
| 8. Split check | Compared class ratios and looked for shared URLs | Same ratio (57.29% / 42.71%) in all sets, and 0 shared URLs |
| 9. Save | Saved the data, the URLs and the feature lists to `data/processed/` | 69 features: 48 original and 21 new |

The data is now clean and ready for model training. Every team member should run this notebook once to create the files in `data/processed/`, because the `data/` folder is not uploaded to GitHub.